Document Processing — Handle both digital PDFs and scanned documents.
Apply OCR for scanned files when needed. Extract clean text and chunk it properly.

Metadata Tagging — Tag chunks with document type, page ranges, and source identifiers. Use consistent metadata for filtering and retrieval.

Embeddings & Indexing — Use an open-source embedding model (e.g., sentence-transformers). Store and retrieve chunks using FAISS or LlamaIndex.

Smart Prompting — Build clear, context-grounded prompts. Instruct the model to cite sources in its answers

Open-Source Model — Use an open-source LLM (Hugging Face, Ollama, etc.) — not Gemini. Ensure the model integrates smoothly with your retrieval logic.

Complete RAG Pipeline — Retrieval → Context Building → Prompt → Model → Answer + Sources. Include confidence scores and chunk count in responses.

User Interface — Build a Gradio UI that allows users to upload documents, view chat history, and see answers with sources and confidence levels. Keep it clean and intuitive — think product demo, not debug tool.

# Part 0 : Install all required libraries ( this will take a while )


In [35]:
!pip install PyMuPDF

!pip install -q pymupdf paddleocr pdf2image pillow numpy
!pip install -q llama-index llama-index-embeddings-huggingface sentence-transformers
!apt-get install -y poppler-utils -q  # required by pdf2image

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


In [36]:
# ============================================
# Import Libraries
# ============================================
import re
import unicodedata
import numpy as np

import fitz  # PyMuPDF
from paddleocr import PaddleOCR
from pdf2image import convert_from_path

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [37]:
from google.colab import files

print("Upload the PDF you want to run through the pipeline:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\nUploaded: {pdf_path}")

Upload the PDF you want to run through the pipeline:


Saving Product Specification Example.pdf to Product Specification Example (2).pdf

Uploaded: Product Specification Example (2).pdf


# Part 1 : Document Processing

## Process Workflow

                 Input PDF
                     │
                     ▼
          ┌──────────────────────┐
          │ Extract Text (PyMuPDF)│
          └──────────────────────┘
                     │
                     ▼
          Is extracted text usable?
               /              \
             Yes              No
              │                │
              ▼                ▼
      Clean & Normalize   Convert PDF to Images
              │                │
              │                ▼
              │         PaddleOCR Processing
              │                │
              └──────────┬─────┘
                         ▼
                 Clean OCR Output
                         ▼
                 Text Chunking
                         ▼
              Embedding Generation
                         ▼
                 Vector Database



In [38]:
# ============================================
# extract_pdf_text()
# ============================================
def extract_pdf_text(pdf_path):
    """
    Extract machine-readable text from a PDF using PyMuPDF.

    Returns
    -------
    str
        Concatenated text from all pages.
    """
    text_parts = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()
    return "\n".join(text_parts)

In [39]:
# ============================================
# is_text_usable()
# ============================================
def is_text_usable(text, min_chars=50, min_words=10, min_printable_ratio=0.85):
    """
    Determine whether extracted text is usable.

    Returns True if:
    - sufficient characters
    - sufficient words
    - reasonable printable ratio
    """
    if text is None:
        return False

    stripped = text.strip()
    if len(stripped) < min_chars:
        return False

    word_count = len(stripped.split())
    if word_count < min_words:
        return False

    printable_count = sum(1 for ch in stripped if ch.isprintable())
    printable_ratio = printable_count / len(stripped) if stripped else 0
    if printable_ratio < min_printable_ratio:
        return False

    return True

In [40]:
# ============================================
# extract_text_with_easyocr()
# ============================================
#
# BUG FIX (debugged Aug 2026): this used to be written against PaddleOCR 2.x's
# API. `pip install paddleocr` (no version pin) now installs PaddleOCR 3.x,
# which changed two things that silently broke this function:
#
#   1. `use_angle_cls=True` is a deprecated alias now. It still works (with a
#      warning), but the current param name is `use_textline_orientation`.
#   2. `.ocr()` is now just a thin deprecated wrapper around `.predict()`, and
#      the return value is no longer a list of [box, (text, score)] pairs per
#      line. Each page result is now a dict-like `OCRResult` object with a
#      'rec_texts' key (list[str] of the recognized lines).
#
#      The old code did `[line[1][0] for line in result[0]]`, which iterates
#      over the DICT KEYS of the new result (e.g. "input_path", "rec_texts",
#      "rec_scores", ...) instead of raising an error. That means every
#      scanned document silently produced a few garbage single characters
#      (e.g. "naotee") instead of real text -- which is exactly the reported
#      symptom, and it either failed the is_text_usable() check downstream or
#      polluted the index with junk instead of the real OCR text.
#
# Fix: parse 'rec_texts' from the dict-like result, with a fallback to the
# old list format in case an older PaddleOCR version ever gets pinned.
!pip install easyocr

import easyocr

print("\nEasyOCR installed and imported!")




EasyOCR installed and imported!


In [41]:
def process_document(pdf_path):
    """
    PyMuPDF first -> check usable -> EasyOCR fallback (on rasterized pages).
    """
    text = extract_pdf_text(pdf_path)

    if is_text_usable(text):
        return text

    # Fallback: rasterize each page to an image, then OCR the images
    import fitz
    reader = easyocr.Reader(['en'])
    doc = fitz.open(pdf_path)
    ocr_text_parts = []
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        results = reader.readtext(img_bytes, detail=0)
        ocr_text_parts.append("\n".join(results))
    doc.close()
    return "\n".join(ocr_text_parts)

In [42]:
!pip install -U paddlepaddle paddleocr

  Using cached paddlepaddle-3.3.1-cp313-cp313-manylinux1_x86_64.whl.metadata (8.8 kB)
  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [43]:
# import sys
# import paddle
# import paddleocr

# print("Python:", sys.version)
# print("Paddle:", paddle.__version__)
# print("PaddleOCR:", paddleocr.__version__)
# print("Paddle location:", paddle.__file__)

In [44]:
import easyocr
import os
# ---- TEST: run extraction end-to-end and sanity-check the output ----
print(pdf_path)
print(os.path.exists(pdf_path))
raw_text = process_document(pdf_path)

print(f"Extracted {len(raw_text)} characters, {len(raw_text.split())} words")
print("-" * 60)
print(raw_text[:500])
print("-" * 60)
assert is_text_usable(raw_text), "Extraction produced unusable text — inspect the PDF/OCR output above."
print("Block 1 passed: raw_text is populated and usable.")

Product Specification Example (2).pdf
True
Extracted 2986 characters, 472 words
------------------------------------------------------------
DFE
pharma
Product Specification
Product group:
Lactose
Brand name:
SuperTab
11SD (EU)
Product code:
743720
Product description:
Lactose monohydrate
Document No_
PD-0072
Page 1 of 2
Manufacturing site:
DFE Pharma GmbH & Co.KG, Norten Hardenberg, Germany
Product name:
SuperTabe 11SD
Conforms to USP
NF, Ph.Eur-, JP, Ch.P. Lactose monohydrate
monograph, current at time of manufacture
Product description:
A white or almost white, crystalline powder freely soluble in water,
practically insoluble in e
------------------------------------------------------------
Block 1 passed: raw_text is populated and usable.


In [45]:
# ============================================
# clean_text()
# ============================================
def clean_text(text):
    """
    Responsibilities:
    - normalize spaces
    - remove duplicated blank lines
    - fix OCR spacing
    - normalize Unicode
    - remove page artefacts
    """
    # normalize Unicode (e.g. curly quotes, ligatures -> standard forms)
    text = unicodedata.normalize("NFKC", text)

    # remove common page artefacts: 'Page 3 of 10', standalone page numbers, form-feed chars
    text = re.sub(r"Page\s+\d+\s+of\s+\d+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*\d+\s*$", " ", text, flags=re.MULTILINE)
    text = text.replace("\x0c", "\n")

    # fix OCR spacing: stray spaces before punctuation, hyphenated line-breaks
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # de-hyphenate wrapped words
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)           # 'word .' -> 'word.'

    # normalize whitespace: collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # remove duplicated blank lines (3+ newlines -> 2)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [46]:
# ---- TEST: show a before/after diff on the raw extraction ----
cleaned_text = clean_text(raw_text)

print("BEFORE (first 300 chars):")
print(repr(raw_text[:500]))
print()
print("AFTER (first 300 chars):")
print(repr(cleaned_text[:500]))
print()
print(f"Length before: {len(raw_text)} | Length after: {len(cleaned_text)}")

BEFORE (first 300 chars):
'DFE\npharma\nProduct Specification\nProduct group:\nLactose\nBrand name:\nSuperTab\n11SD (EU)\nProduct code:\n743720\nProduct description:\nLactose monohydrate\nDocument No_\nPD-0072\nPage 1 of 2\nManufacturing site:\nDFE Pharma GmbH & Co.KG, Norten Hardenberg, Germany\nProduct name:\nSuperTabe 11SD\nConforms to USP\nNF, Ph.Eur-, JP, Ch.P. Lactose monohydrate\nmonograph, current at time of manufacture\nProduct description:\nA white or almost white, crystalline powder freely soluble in water,\npractically insoluble in e'

AFTER (first 300 chars):
'DFE\npharma\nProduct Specification\nProduct group:\nLactose\nBrand name:\nSuperTab\n11SD (EU)\nProduct code:\n \nProduct description:\nLactose monohydrate\nDocument No_\nPD-0072\n \nManufacturing site:\nDFE Pharma GmbH & Co.KG, Norten Hardenberg, Germany\nProduct name:\nSuperTabe 11SD\nConforms to USP\nNF, Ph.Eur-, JP, Ch.P. Lactose monohydrate\nmonograph, current at time of manufacture\nProduct description:\nA white

# Part 2: Metadata Tagging


- perform text tagging for more fast and accurate retrival
- perform chuncking

In [ ]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================
# Install llama-cpp-python pre-built wheel with CUDA support for fast GPU inference
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# Install PyMuPDF (fitz) for PDF text extraction and pandas for tabular output
!pip install -q PyMuPDF pandas

# ============================================================
# STEP 1: Download Mistral-7B-Instruct-v0.2 (GGUF Quantized)
# ============================================================
import os

model_path = "/content/mistral-7b-instruct-v0.2.Q4_K_M.gguf"

if not os.path.exists(model_path):
    print("Downloading Mistral 7B GGUF model...")
    !wget -c https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print("Download completed successfully.")
else:
    print(f"Model already exists at: {model_path}")

# ============================================================
# STEP 3: Initialize Mistral 7B Engine
# ============================================================
from llama_cpp import Llama

# Load model and offload all layers to GPU VRAM for fast zero-shot inference
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # Offload all 33 layers to GPU VRAM
    n_ctx=4096,           # Context window
    temperature=0.0,      # Deterministic classification output
    verbose=False
)

def run_mistral(prompt: str, max_tokens: int = 120) -> str:
    """Helper function to format prompts with Mistral instruction tags."""
    formatted_prompt = f"[INST] {prompt.strip()} [/INST]"
    output = llm(
        formatted_prompt,
        max_tokens=max_tokens,
        stop=["</s>", "\n\n\n"],
        echo=False
    )
    return output["choices"][0]["text"].strip()

print("Mistral 7B loaded and ready.")

In [ ]:
# ============================================================
# STEP 5: Classification and Boundary Detection Function
# ============================================================
import json
import re

ALLOWED_DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other".
    "Product Specification"
]

def analyze_page_boundary_and_type(page_idx: int, curr_text: str, prev_text: str = None, current_doc_type: str = None) -> dict:
    """
    Evaluates whether a page starts a new document or continues the previous one,
    and assigns the standardized document type.
    """
    if page_idx == 0:
        prompt = f"""You are a pharmaceutical document classifier.
Classify the following text into EXACTLY ONE category from this list:
- Cover Letter
- Certificate of Quality
- Packaging Specification
- BSE/TSE Declaration
- Material Description
- Supplier Qualification
- Chain of Custody
- Product Specification
- Other

Page Content:
\"\"\"{curr_text[:1200]}\"\"\"

Respond ONLY with a valid JSON object in this exact format:
{{"is_new_doc": "Yes", "doc_type": "<Doc Type>"}}"""
    else:
        prompt = f"""You are analyzing consecutive pages from a bundled pharmaceutical PDF.
Determine if the Current Page starts a NEW document or CONTINUES the previous document.

A page starts a NEW document ("is_new_doc": "Yes") if:
- It contains a new document title or header (e.g., "Certificate of Quality", "Packaging Specification").
- It contains a different lot number, revision number, or new product name.
- It is a standalone certificate/declaration.

A page CONTINUES the previous document ("is_new_doc": "No") if:
- It explicitly states "continued", "page 2 of 2", or has matching document control numbers.
- It is a continuation of tables/sections from the previous page.

Previous Document Type: {current_doc_type}
Previous Page Excerpt:
\"\"\"{prev_text[:600]}\"\"\"

Current Page Content:
\"\"\"{curr_text[:1200]}\"\"\"

If it is a new document, choose doc_type from:
["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration", "Material Description", "Supplier Qualification", "Chain of Custody", "Other"].
If it is NOT a new document, keep the previous doc_type: "{current_doc_type}".

Check all the page and find the Issue Date of this document Valid Date for this document , the date format should be all standardized to YYYY-MM-DD. If the date is not avaliable ot not found, enter default 0000-00-00


Respond ONLY with a valid JSON object in this exact format:
{{"is_new_doc": "Yes" or "No", "doc_type": "<Doc Type>"}}"""

    raw_response = run_mistral(prompt, max_tokens=80)

    # Fallback and JSON parsing handling
    try:
        json_match = re.search(r"\{.*?\}", raw_response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group(0))
            is_new = parsed.get("is_new_doc", "Yes").strip().capitalize()
            doc_type = parsed.get("doc_type", "Other").strip()
        else:
            raise ValueError("No JSON found in response")
    except Exception:
        is_new = "Yes" if "yes" in raw_response.lower() else "No"
        doc_type = current_doc_type if is_new == "No" and current_doc_type else "Other"

    # Match against standardized allowed types
    matched_type = "Other"
    for allowed in ALLOWED_DOC_TYPES:
        if allowed.lower() in doc_type.lower():
            matched_type = allowed
            break

    return {
        "is_new_doc": "Yes" if is_new.startswith("Y") else "No",
        "doc_type": matched_type
    }

In [ ]:
# ============================================================
# STEP 6: Execute Pipeline Loop and Track page_in_doc
# ============================================================
pipeline_results = []
current_doc_type = None
page_in_doc_counter = 0

for i, page in enumerate(doc_pages):
    curr_text = page["text"]
    prev_text = doc_pages[i - 1]["text"] if i > 0 else None

    analysis = analyze_page_boundary_and_type(
        page_idx=i,
        curr_text=curr_text,
        prev_text=prev_text,
        current_doc_type=current_doc_type
    )

    # Manage page_in_doc index tracking
    if analysis["is_new_doc"] == "Yes":
        page_in_doc_counter = 0
        current_doc_type = analysis["doc_type"]
    else:
        page_in_doc_counter += 1
        analysis["doc_type"] = current_doc_type

    pipeline_results.append({
        "page": i,
        "is_new_doc": analysis["is_new_doc"],
        "doc_type": analysis["doc_type"],
        "page_in_doc": page_in_doc_counter
    })

    print(f"Page {i:02d} -> is_new_doc: {analysis['is_new_doc']}, doc_type: {analysis['doc_type']}, page_in_doc: {page_in_doc_counter}")

In [ ]:
# ============================================================
# STEP 7: Eval: Format and Display Final Results
# ============================================================
import pandas as pd

# Display Pandas DataFrame
df_results = pd.DataFrame(pipeline_results)
print("=== Final Metadata DataFrame ===")
display(df_results)

# Display Structured JSON Output
print("\n=== Final JSON Metadata Output ===")
print(json.dumps(pipeline_results, indent=2))

In [48]:

"""
	Create one Document per page/section
   (each carrying its own metadata dict), then run SentenceSplitter on that list
"""
def chunk_document(pages, chunk_size=512, overlap=100, source="document"):
    """
    pages: list of dicts like {"text": "...", "page_number": 1, "section": "Specifications"}
    """
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=overlap)

    documents = [
        Document(
            text=p["text"],
            metadata={"source": source, "page": p["page_number"], "section": p.get("section")}
        )
        for p in pages
    ]
    text_nodes = splitter.get_nodes_from_documents(documents)

    return [
        {"chunk_id": i, "source": source, "text": n.get_content(), **n.metadata}
        for i, n in enumerate(text_nodes)
    ]

In [56]:
# ---- TEST: confirm chunk count, overlap, and structure look right ----
nodes = chunk_document(cleaned_text, chunk_size=512, overlap=100, source=pdf_path)

print(f"Created {len(nodes)} chunks")
print("-" * 60)
print("Sample node[0]:")
print(nodes[0])
print("-" * 60)
if len(nodes) > 1:
    print("End of chunk 0:", nodes[0]["text"][-100:])
    print("Start of chunk 1:", nodes[1]["text"][:100])
    print("(the overlapping words above should roughly match — that's chunk_overlap=100 at work)")

print(nodes[1])

Created 3 chunks
------------------------------------------------------------
Sample node[0]:
{'chunk_id': 0, 'source': 'Product Specification Example (2).pdf', 'text': 'DFE\npharma\nProduct Specification\nProduct group:\nLactose\nBrand name:\nSuperTab\n11SD (EU)\nProduct code:\n \nProduct description:\nLactose monohydrate\nDocument No_\nPD-0072\n \nManufacturing site:\nDFE Pharma GmbH & Co.KG, Norten Hardenberg, Germany\nProduct name:\nSuperTabe 11SD\nConforms to USP\nNF, Ph.Eur-, JP, Ch.P. Lactose monohydrate\nmonograph, current at time of manufacture\nProduct description:\nA white or almost white, crystalline powder freely soluble in water,\npractically insoluble in ethanol\nResidual solvents\n(CPMPIICHI283/95):\nNo class 1,2,3 solvents are used during production\nPhysical-Chemical data:\nSpecification\nIdentification:\nComplies with Pharmacopoeia when tested\nAppearance of solution (Ph.Eur:)\nClear and not more coloured than ref:\nClarity and Colour of Solution\nClear and colourles

## Embeddings

In [50]:
# ============================================
# embed_chunks()
# ============================================
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def embed_chunks(nodes, embed_model=embed_model):
    """
    Compute an embedding vector for every chunk's text.

    Returns
    -------
    list[dict]
        Same nodes, each with an added 'embedding' key (list[float]).
    """
    texts = [n["text"] for n in nodes]
    vectors = embed_model.get_text_embedding_batch(texts, show_progress=True)

    for node, vector in zip(nodes, vectors):
        node["embedding"] = vector

    return nodes

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
# ---- TEST: confirm every chunk got a same-length vector ----
embedded_nodes = embed_chunks(nodes)

dims = {len(n["embedding"]) for n in embedded_nodes}
print(f"Embedded {len(embedded_nodes)} chunks")
print(f"Embedding dimension(s) seen: {dims}")
print("First 5 values of chunk 0's embedding:", embedded_nodes[0]["embedding"][:5])
assert len(dims) == 1, "All chunks should share the same embedding dimension."
print("Block 3 passed: embeddings are populated and consistent.")

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Embedded 3 chunks
Embedding dimension(s) seen: {384}
First 5 values of chunk 0's embedding: [-0.0106429448351264, -0.15067481994628906, -0.020097780972719193, -0.04972146078944206, 0.06795398145914078]
Block 3 passed: embeddings are populated and consistent.


In [52]:
# ============================================
# retrieve_top_k() — cosine-similarity search over embedded_nodes
# ============================================
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve_top_k(query, embedded_nodes, embed_model=embed_model, top_k=3):
    query_vector = embed_model.get_query_embedding(query)
    scored = [
        (cosine_similarity(query_vector, n["embedding"]), n)
        for n in embedded_nodes
    ]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

In [53]:
# ---- TEST: replace this with a question relevant to your uploaded PDF ----
test_query = "What are the storage condition?"
results = retrieve_top_k(test_query, embedded_nodes, top_k=3)

print(f"Query: {test_query}\n")
for rank, (score, node) in enumerate(results, start=1):
    print(f"#{rank} | similarity={score:.3f} | chunk_id={node['chunk_id']}")
    print(node["text"].replace(chr(10), ' '))
    print("-" * 60)

Query: What are the storage condition?

#1 | similarity=0.219 | chunk_id=2
Box 20 21.20, 47568 Goch, Germany, T. +49 2823 9288 770, F. +49 2823 9288 7799 Edition No.   Issue date: 13 Feb 2025 Valid until: 13 Feb 2028
------------------------------------------------------------
#2 | similarity=0.090 | chunk_id=0
DFE pharma Product Specification Product group: Lactose Brand name: SuperTab 11SD (EU) Product code:   Product description: Lactose monohydrate Document No_ PD-0072   Manufacturing site: DFE Pharma GmbH & Co.KG, Norten Hardenberg, Germany Product name: SuperTabe 11SD Conforms to USP NF, Ph.Eur-, JP, Ch.P. Lactose monohydrate monograph, current at time of manufacture Product description: A white or almost white, crystalline powder freely soluble in water, practically insoluble in ethanol Residual solvents (CPMPIICHI283/95): No class 1,2,3 solvents are used during production Physical-Chemical data: Specification Identification: Complies with Pharmacopoeia when tested Appearance of

In [54]:
# Run this in a Colab notebook or terminal
!pip install --upgrade gradio

# 💡 Tip: Use a virtual environment for local installs if you're working locally


#Part 7: User interface

The following gradio UI has the main features. that are covered in the instructions

1. Upload documents
2. View Chat History ( export in .txt form ). DONE
3. Answer and source with confidence score


Minor Feature for better UI
1. Allow multiple folder uploads
2. loading indicator while processing

In [55]:
import gradio as gr
import fitz  # PyMuPDF
from google.colab import userdata
import os
import tempfile

# Holds extracted syllabus text so handle_chat can use it as context
document_context = ""

# Document processing
def process_pdfs(files):
    global document_context

    if not files:
        return "⚠️ No files uploaded yet."

    summaries = []
    combined_text = []
    for f in files:
        doc = fitz.open(f.name)
        text = "\n".join(page.get_text() for page in doc)
        filename = f.name.split("/")[-1]
        summaries.append(f"📄 {filename}: {len(doc)} page(s), {len(text)} characters extracted")
        combined_text.append(f"--- {filename} ---\n{text}")
        doc.close()

    document_context = "\n\n".join(combined_text)
    return "✅ Processed " + str(len(files)) + " file(s):\n" + "\n".join(summaries)


# Chat handling
def handle_chat(message, history):
    global document_context

    if not message.strip():
        yield history, ""
        return

    if not document_context:
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Please upload and process a syllabus PDF first."},
        ]
        yield history, ""
        return

    # 1. Show the user's message with a loading placeholder immediately
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": "⏳ Generating answer..."},
    ]
    yield history, ""

    # 2. Send the processed document + user question to Groq
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant answering student questions about a course syllabus. "
                    "Only use the provided syllabus content to answer. If the answer isn't in the "
                    "syllabus, say so instead of guessing.\n\n"
                    f"Syllabus content:\n{document_context}"
                ),
            },
            {"role": "user", "content": message},
        ],
    )
    answer = response.choices[0].message.content

    # 3. Swap the placeholder with Groq's real answer, which Gradio then displays
    history[-1] = {"role": "assistant", "content": answer}
    yield history, ""


# ---------------------------------------------------------------------------
# NEW FEATURE 1: Light / Dark theme toggle
# ---------------------------------------------------------------------------
# Gradio locks its theme at launch time, so a runtime toggle is done with a
# small CSS block (two variants scoped under a `.dark-mode` class) plus a
# button that flips that class via JS. No server round-trip needed.

toggle_theme_js = """
() => {
    const shell = document.getElementById('app-shell');
    if (shell) { shell.classList.toggle('dark-mode'); }
}
"""


# ---------------------------------------------------------------------------
# NEW FEATURE 2: Export chat history as .txt
# ---------------------------------------------------------------------------
def export_chat(history):
    """Writes the conversation to a temp .txt file in the required format:
    User: [User query]
    Response: [The model response]
    """
    import tempfile  # local import so this works even if the cell is re-run out of order

    if not history:
        gr.Warning("There's no chat history yet to export.")
        return None

    lines = []
    for i in range(0, len(history) - 1, 2):
        user_turn = history[i]
        bot_turn = history[i + 1]
        user_msg = user_turn.get("content", "")
        bot_msg = bot_turn.get("content", "")
        lines.append(f"User: {user_msg}")
        lines.append(f"Response: {bot_msg}")

    content = "\n".join(lines)

    tmp = tempfile.NamedTemporaryFile(
        mode="w", suffix=".txt", delete=False, encoding="utf-8"
    )
    tmp.write(content)
    tmp.close()
    return tmp.name


with gr.Blocks(title="Course Syllabus Assistant") as demo:
    with gr.Column(elem_id="app-shell"):
        with gr.Row():
            gr.Markdown("### 📚 Course Syllabus Assistant")
            theme_btn = gr.Button("🌙 / ☀️ Toggle Theme", scale=0)

        with gr.Row():
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label="Chat History", height=400)
                user_input = gr.Textbox(
                    placeholder="Ask a question about your syllabus...",
                    label="Your Question",
                )
                with gr.Row():
                    send_btn = gr.Button("📤 Send")
                    clear_btn = gr.Button("🗑️ Clear Chat")
                    export_btn = gr.DownloadButton("💾 Export Chat (.txt)")

            with gr.Column(scale=1):
                pdf_input = gr.File(
                    label="📄 Upload Syllabus PDF(s)",
                    file_types=[".pdf"],
                    file_count="multiple",
                )
                process_btn = gr.Button("🔄 Process Documents")
                status_box = gr.Textbox(label="Status", interactive=False, lines=6)

        process_btn.click(process_pdfs, inputs=pdf_input, outputs=status_box)
        send_btn.click(handle_chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
        user_input.submit(handle_chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
        clear_btn.click(lambda: [], outputs=chatbot)

        # Feature 1: theme toggle (pure client-side JS, no Python callback)
        theme_btn.click(None, None, None, js=toggle_theme_js)

        # Feature 2: export chat history to .txt
        export_btn.click(export_chat, inputs=chatbot, outputs=export_btn)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://917af1edd95e9de08c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://917af1edd95e9de08c.gradio.live
